#### x[crop, market] = quantity allocated
#### maximize expected profit
#### (expected_price - purchase_cost - transport_cost) × quantity

In [ ]:
import pulp

crops = model_data["crop"].unique()
markets = model_data["market"].unique()

# Create optimization model
model = pulp.LpProblem("Crop_Distribution", pulp.LpMaximize)

# Decision variables
decision_vars = pulp.LpVariable.dicts(
    "allocation",
    [(c, m) for c in crops for m in markets],
    lowBound=0
)

# Objective function
model += pulp.lpSum(
    (model_data.loc[(model_data.crop == c) &
                    (model_data.market == m),
                    "expected_price"].values[0]
     - model_data.loc[(model_data.crop == c) &
                      (model_data.market == m),
                      "transport_cost"].values[0])
    * decision_vars[(c, m)]
    for c in crops for m in markets
)

In [ ]:
for c in crops:
    available_supply = supply.loc[supply.crop == c, "supply_available"].values[0]

    model += pulp.lpSum(
        decision_vars[(c, m)] for m in markets
    ) <= available_supply

In [ ]:
for m in markets:
    demand_limit = forecast_demand.loc[
        forecast_demand.market == m,
        "demand_forecast"
    ].values[0]

    model += pulp.lpSum(
        decision_vars[(c, m)] for c in crops
    ) <= demand_limit

In [ ]:
for m in markets:
    capacity = transport.loc[
        transport.market == m,
        "transport_capacity"
    ].values[0]

    model += pulp.lpSum(
        decision_vars[(c, m)] for c in crops
    ) <= capacity

In [ ]:
model.solve()

print("Status:", pulp.LpStatus[model.status])

In [ ]:
results = []

for v in model.variables():
    if v.varValue > 0:
        crop, market = v.name.split("_")[1:]
        results.append({
            "crop": crop,
            "market": market,
            "quantity": v.varValue
        })

allocation_df = pd.DataFrame(results)

allocation_df.to_csv("optimal_allocation.csv", index=False)

print(allocation_df)

In [ ]:
scenario = model_data.copy()

scenario["expected_price"] = scenario["expected_price"] * 0.9

In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="password",
    database="agriculture_db"
)

cursor = conn.cursor()

for _, row in allocation_df.iterrows():
    query = """
    INSERT INTO optimization_results
    (crop, market, quantity)
    VALUES (%s, %s, %s)
    """
    cursor.execute(query, (row.crop, row.market, row.quantity))

conn.commit()
conn.close()

In [ ]:
improvement = optimized_profit - baseline_profit
print("Profit improvement:", improvement)